<a id="inicio"></a>

# Book Recommender System

## Objective

Build a content-based book recommendation engine that combines classical IR, modern embeddings, and collaborative filtering into a hybrid recommender.

## Methodology

- **Content-based similarity with TF-IDF**: Represent each book as a bag-of-words vector over cleaned, lemmatized text
- **Content-based similarity with embeddings**: Use sentence embeddings (via ChromaDB) to capture semantic similarity beyond term overlap
- **Personalized recommendation**: For a sample user, retrieve books similar to their highly-rated items, deduplicate and rank by community rating
- **Hybrid recommendation**: Combine content-based candidate retrieval with collaborative-filtering scores from an SVD matrix factorization model (`surprise` library)

## Source datasets

From [goodbooks-10k](https://www.kaggle.com/zygmunt/goodbooks-10k) on Kaggle:
- `books.csv` — 10,000 books with metadata
- `ratings.csv` — ~1M ratings from 53k+ users
- `tags.csv` — tag dictionary
- `book_tags.csv` — per-book tag assignments
- `overviews/*.txt` — plot summaries scraped from Goodreads

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. Clone the repo and create a Python 3.9+ virtual environment:
   `python -m venv .venv && source .venv/bin/activate`
2. Install dependencies including `spacy`, `scikit-learn`, `chromadb`, and `scikit-surprise`:
   `pip install -r requirements.txt`
3. Download the spaCy English model: `python -m spacy download en_core_web_sm`
4. Place the goodbooks-10k CSVs and `overviews/` folder under `data/`.
5. Launch Jupyter:
   `jupyter lab` (or `jupyter notebook`), open this file, select the venv kernel, and run every cell top-to-bottom.

</div>

---

---

<a id="indice"></a>
## Contents

* [1. Introduction](#section1)
   * [Loading the reviews](#section11)
   * [Tags](#section12)
* [2. Similarity Search with TF-IDF](#section2)
    * [Text cleaning and preparation](#section21)
    * [Similarity search](#section22)
* [3. Similarity Search with Embeddings](#section3)
* [4. Content-Based Recommendation](#section4)
   * [Selecting the user's preferred books](#section41)
   * [Finding similar books](#section42)
   * [Ranking and final recommendation](#section43)
* [5. Hybrid Recommender System](#section5)

---

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

import warnings
warnings.filterwarnings('ignore')

<a id="section1"></a>
## 1. Introduction

Collaborative filtering recommenders rely purely on the user-item rating matrix — they ignore content entirely. While this works well for popular items with many ratings, recommendations can feel disconnected from what the user was actually reading.

This project builds a content-based recommender on top of the [goodbooks-10k](https://www.kaggle.com/zygmunt/goodbooks-10k) dataset, then layers an embedding-based variant (using modern sentence-embedding techniques) and finally a hybrid that also pulls in collaborative filtering scores via SVD matrix factorization.

The dataset contains 10,000 books scraped from [Goodreads](http://goodreads.com), complete with ratings and tags from 53,000+ users.

<div class="alert alert-block alert-warning">
⚠️ The original dataset has been lightly cleaned (normalized identifiers, simplified indexing) to make it easier to work with.
</div>

In [3]:
import pandas as pd
import numpy as np

df_goodreads = pd.read_csv('data/books.csv', sep="\t")
df_goodreads.head(2)

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section11"></a>
### Loading the reviews

The main table carries very little content information about each book. Fortunately, plot summaries can be scraped from Goodreads and saved as plain text files. Each book's summary lives in `./data/overviews/<gr_book_id>.txt`, keyed by the book's Goodreads ID.

### Reading overview files

Implement `get_overview(gr_book_id)` that reads the summary text file for a book ID and returns the contents as a string, or `None` if the file doesn't exist or is empty.

In [ ]:
def get_overview(gr_book_id):
    import os
    path = f'./data/overviews/{gr_book_id}.txt'
    if not os.path.isfile(path):
        return None
    with open(path, 'r', encoding='utf-8') as f:
        overview = f.read().strip()
    return overview if overview else None

    
get_overview(320) 

'(Book Jacket Status: Jacketed)The brilliant, bestselling, landmark novel that tells the story of the Buendia family, and chronicles the irreconcilable conflict between the desire for solitude and the need for love—in rich, imaginative prose that has come to define an entire genre known as "magical realism."'

Create an `overview` column on `df_goodreads` populated with each book's summary. Fill missing values with an empty string.

In [5]:
df_goodreads['overview'] = df_goodreads['gr_book_id'].apply(lambda x: get_overview(x) or "")


For readability, keep only the relevant columns and set `gr_book_id` as the index.

In [6]:
columns = ['gr_book_id','authors', 'original_publication_year', 'title', 'average_rating','overview']
df_goodreads = df_goodreads[columns].set_index('gr_book_id')
df_goodreads.head(2)

,authors,original_publication_year,title,average_rating,overview
gr_book_id,,,,,
2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",4.34,Winning will make you famous. Losing means cer...
3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,4.44,Harry Potter's life is miserable. His parents ...


Drop rows whose `overview` is empty.

In [7]:
df_goodreads = df_goodreads[df_goodreads['overview'] != ""]


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section12"></a>
### Tags

The dataset includes user-contributed tags. Two files are relevant: `tags.csv` maps tag IDs to tag names, and `book_tags.csv` assigns tags to books. Load them into `df_tags` and `df_book_tags`.

In [8]:
df_tags = pd.read_csv('./data/tags.csv')
df_book_tags = pd.read_csv('./data/book_tags.csv')

print('Etiquetas', end='')
display(df_tags.iloc[2000:2002])

print('\n Etiquetas por libro', end='')
display(df_book_tags.head(3))

Etiquetas

,tag_id,tag_name
2000,2000,alex-read
2001,2001,alex-rider



 Etiquetas por libro

,gr_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173


Join both tables on tag ID to produce a unified `df_book_tags` with human-readable tag names.

In [9]:
df_book_tags=df_book_tags.merge(df_tags, left_on='tag_id', right_on="tag_id").drop(columns=['count', 'tag_id'])
df_book_tags.tail()

,gr_book_id,tag_name
999907,33288638,neighbors
999908,33288638,kindleunlimited
999909,33288638,5-star-reads
999910,33288638,fave-author
999911,33288638,slowburn


Inspect the tag distribution — which tags are most common, and are any so rare they should be dropped?

In [10]:
etiquetas_conteo = df_book_tags['tag_name'].value_counts()
etiquetas_conteo


tag_name
to-read               9983
favorites             9881
owned                 9858
books-i-own           9799
currently-reading     9776
                      ... 
prose-literature         1
best-reads-2016          1
on-the-stack             1
beautiful-pictures       1
slowburn                 1
Name: count, Length: 34252, dtype: int64

Many tags appear only once or twice and offer no discriminative signal. Drop them.

Drop tags that appear fewer than 20 times.

<div class="alert alert-block alert-warning">
⚠️ There are multiple ways to do this. One clean option: group by `tag_name` with `groupby` and use `filter` to keep only groups with size ≥ 20.
</div>

In [12]:
df_book_tags = df_book_tags.groupby('tag_name').filter(lambda x: len(x) >= 20)



Another problem: some tags are generic (reading intent or personal annotations) and carry no information about the book itself — e.g. `read-readings` or `favourites`.

In [ ]:
# TODO: Write a filter/regex expression to drop generic tags like 'read-readings',
# 'favourites', 'to-read', etc. One approach: use Series.str.contains with a
# joined regex pattern from the target_tags list.
# df_book_tags = df_book_tags[~df_book_tags['tag_name'].str.contains('|'.join(target_tags), regex=True)]


Drop tags that match the blocklist in `target_tags`.

<div class="alert alert-block alert-warning">
⚠️ Match by substring (not exact match) so that tag variants are also dropped. `Series.str.contains` accepts regular expressions.
</div>

In [13]:
target_tags = ['read', 'favo', 'own', 'top', 'book', 'librar', 'kindle', 'list', 'to_buy', 'current']
pattern = '|'.join(target_tags)

df_book_tags = df_book_tags[~df_book_tags['tag_name'].str.contains(pattern, case=False, regex=True)]

    
df_book_tags

,gr_book_id,tag_name
1,1,fantasy
4,1,young-adult
5,1,fiction
6,1,harry-potter
9,1,ya
...,...,...
999899,33288638,reviewed
999902,33288638,stand-alones
999903,33288638,alpha
999904,33288638,kick-ass-heroine


Build `df_book_tag_text`: for each book (keyed by `goodreads_book_id`), concatenate all its tags into a single text field. Group `df_book_tags` by book ID and `join` the tag names.

In [14]:
df_book_tag_text = (df_book_tags.groupby('gr_book_id')['tag_name']
                                .apply(lambda group_tags: f"{' '.join(group_tags):s}")
                                .to_frame()
                                .rename(columns={'tag_name':'tags'}))
        
df_book_tag_text.head()

,tags
gr_book_id,
1,fantasy young-adult fiction harry-potter ya se...
2,fantasy children children-s default fiction yo...
3,fantasy young-adult fiction harry-potter ya se...
5,fantasy young-adult fiction harry-potter ya se...
6,fantasy young-adult fiction harry-potter ya se...


Merge the tag text column back into the main `df_goodreads` DataFrame.

In [ ]:
df_goodreads = df_goodreads.merge(df_book_tag_text, left_index=True, right_index=True, how='left')


In [27]:
df_goodreads = df_goodreads.drop(columns=['tags_y'])  


In [28]:
df_goodreads.head()

,authors,original_publication_year,title,average_rating,overview,tags
gr_book_id,,,,,,
2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",4.34,Winning will make you famous. Losing means cer...,young-adult fiction fantasy ya science-fiction...
3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,4.44,Harry Potter's life is miserable. His parents ...,fantasy young-adult fiction harry-potter ya se...
41865,Stephenie Meyer,2005.0,"Twilight (Twilight, #1)",3.57,About three things I was absolutely positive.F...,young-adult fantasy vampires ya fiction parano...
2657,Harper Lee,1960.0,To Kill a Mockingbird,4.25,The unforgettable novel of a childhood in a sl...,classics classic historical-fiction school clà...
4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,3.89,"On its first publication in 1925, The Great Ga...",classics fiction classic literature school his...


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section2"></a>
## 2. Similarity Search with TF-IDF

TF-IDF is a classic way to represent documents for comparison. Each book becomes a weighted term vector, and similarity is computed via cosine similarity between those vectors.

<a id="section21"></a>
### Text cleaning and preparation

TF-IDF uses a bag-of-words model, which benefits from aggressive text preprocessing — lemmatization, lowercasing, stopword removal, and dropping non-alphanumeric tokens. We use `spaCy` with the `en_core_web_sm` model — not the highest-performing English model, but efficient and sufficient here (the transformer-based `en_core_web_trf` would be slower).

In [ ]:
#! python -m spacy download es_dep_news_trf
#! python -m spacy download en_core_web_sm # Solo se utilizará este
#! python -m spacy download en_core_web_trf 

In [29]:
import spacy 

# Crea un objeto con el pipeline
nlp = spacy.load("en_core_web_sm")
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

### Text cleaning function

Implement `clean(text)` that:
- Runs the input through spaCy
- Filters out non-alphanumeric tokens and stopwords
- Returns a space-separated string of lowercased lemmas

In [30]:
def clean(overview):
    doc = nlp(overview)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]
    return " ".join(tokens)


overview = df_goodreads.iloc[0]['overview']
clean(overview)

'win famous lose mean certain death nation panem form post apocalyptic north america country consist wealthy capitol region surround poor district early history rebellion lead district capitol result destruction creation annual televise event know hunger games punishment reminder power grace capitol district yield boy girl age lottery system participate game tribute choose annual reaping force fight death leave survivor claim victory year old katniss young sister prim select district female representative katniss volunteer place male counterpart peeta pit big strong representative train life see death sentence katniss close death survival second nature'

Build a `text_books` Series by cleaning each book's `overview` and concatenating it with the book's `tags` (space-separated).

<div class="alert alert-block alert-warning">
⚠️ This cell takes about five minutes to run. It's written this way (without `nlp.pipe` batching) for clarity — in this case the runtime difference is negligible. Because the result is only used in this section, we don't add it back to the main DataFrame.
</div>

In [31]:
text_books = df_goodreads.apply(
    lambda row: f"{clean(row['overview'])} {row['tags']}", axis=1
)


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section22"></a>
### Similarity search

### Building the TF-IDF matrix

Create a `TfidfVectorizer` named `tfidf_vect` with a max of 20,000 features. Fit it on `text_books` and store the resulting matrix in `text_books_tfidf`. Save the vocabulary in a `terms` variable for inspection later.

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(max_features=20000)
text_books_tfidf = tfidf_vect.fit_transform(text_books)
terms = tfidf_vect.get_feature_names_out()

Pick a sample book from `df_goodreads` — we'll use *Twilight*.

In [33]:
twilight_id = 41865
twilight_pos = df_goodreads.index.get_loc(twilight_id)

print(df_goodreads.loc[twilight_id])
text_query = df_goodreads.loc[twilight_id]['overview']
text_query

authors                                                        Stephenie Meyer
original_publication_year                                               2005.0
title                                                  Twilight (Twilight, #1)
average_rating                                                            3.57
overview                     About three things I was absolutely positive.F...
tags                         young-adult fantasy vampires ya fiction parano...
Name: 41865, dtype: object


"About three things I was absolutely positive.First, Edward was a vampire.Second, there was a part of him—and I didn't know how dominant that part might be—that thirsted for my blood.And third, I was unconditionally and irrevocably in love with him.In the first book of the Twilight Saga, internationally bestselling author Stephenie Meyer introduces Bella Swan and Edward Cullen, a pair of star-crossed lovers whose forbidden relationship ripens against the backdrop of small-town suspicion and a mysterious coven of vampires. This is a love story with bite."

### Encoding a query

To search by similarity, the query text has to be transformed into the same TF-IDF space. It needs the same cleaning (`clean()`) and transformation (`tfidf_vect.transform()`) as the corpus.

Transform `text_query` into `text_query_tfidf`.

<div class="alert alert-block alert-warning">
⚠️ This is the general approach for *new* queries. Since our query book already lives in the corpus, we could also just look up its row in `text_books_tfidf` directly — but we want to demonstrate the full pipeline.
</div>

In [ ]:
text_query_cleaned = clean(text_query)

text_query_tfidf = tfidf_vect.transform([text_query_cleaned])


Show the 10 highest-TF-IDF terms for the query document.

In [ ]:
import numpy as np

tfidf_array = text_query_tfidf.toarray().flatten()

top_indices = np.argsort(tfidf_array)[::-1][:10]

top_terms = [(terms[i], tfidf_array[i]) for i in top_indices]

for term, value in top_terms:
    print(f"{term}: {value:.4f}")


edward: 0.3174
unconditionally: 0.2304
ripen: 0.2304
stephenie: 0.2092
vampire: 0.2060
coven: 0.2027
cullen: 0.2009
dominant: 0.1918
meyer: 0.1918
bella: 0.1894


### Cosine similarity

Compute cosine similarities between `text_query_tfidf` and every row of `text_books_tfidf`. Store the result in a 1-D vector `similarities_tfidf`.

<div class="alert alert-block alert-warning">
⚠️ `cosine_similarity` returns a matrix even for a single query. Extract row 0 (or flatten the result) to get a 1-D vector for easier downstream operations.
</div>

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

similarities_matrix = cosine_similarity(text_query_tfidf, text_books_tfidf)

similarities_tfidf = similarities_matrix[0]

print(similarities_tfidf[:10])


[0.01781354 0.00527974 0.72253601 0.02319492 0.01892415 0.02201402
 0.         0.01358402 0.00981124 0.00784296]


### Top-6 similar books

Get the indices of the 6 most similar books (the most similar will obviously be the query book itself).

<div class="alert alert-block alert-warning">
⚠️ Remember that DataFrame indices aren't the same as row positions. `similarities_tfidf` is a NumPy array, so use integer positions — then look up the actual book IDs in the DataFrame afterward.
</div>

In [37]:
top_similarities = np.argsort(similarities_tfidf)[::-1][:6]

df_goodreads.iloc[top_similarities]['title']


gr_book_id
41865                                Twilight (Twilight, #1)
3090465                   The Twilight Saga (Twilight, #1-4)
6441654    Twilight and Philosophy: Vampires, Vegetarians...
7140220                                 Twilight and History
49041                                New Moon (Twilight, #2)
1162543                         Breaking Dawn (Twilight, #4)
Name: title, dtype: object

The results make sense — all five neighbors come from the Twilight saga. This is driven by highly specific terms like "Twilight" dominating the similarity scores.

### Second query — *Animal Farm*

Run the same search for `gr_book_id = 7613` (Orwell's *Animal Farm*).

In [38]:
animalfarm_id = 7613

overview = df_goodreads.loc[animalfarm_id]['overview']
tags = df_goodreads.loc[animalfarm_id]['tags']

text_query = clean(overview) + ' ' + tags

text_query_tfidf = tfidf_vect.transform([text_query])

similarities_tfidf = cosine_similarity(text_query_tfidf, text_books_tfidf)[0]

top_similarities = np.argsort(similarities_tfidf)[::-1][:6]

df_goodreads.iloc[top_similarities]['title']


gr_book_id
7613                    Animal Farm
5472             Animal Farm / 1984
7624              Lord of the Flies
32650    The Return of the Native  
3102                    Howards End
5344                     Hard Times
Name: title, dtype: object

Some recommendations are clearly related to *Animal Farm*'s themes; others aren't. TF-IDF handles specific proper nouns well but struggles with thematic abstraction.

<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section3"></a>
## 3. Similarity Search with Embeddings

Modern language models (transformers) have changed the game for text similarity. Where classical embeddings could only encode individual words, today's sentence-level models can encode full paragraphs while capturing semantics far richer than term overlap.

In this section we use sentence embeddings to represent each book's summary and perform similarity search. Rather than managing embeddings ourselves, we use **ChromaDB** — a vector database with built-in support for automatic embedding, storage, and similarity queries.

<div class="alert alert-block alert-warning">
⚠️ ChromaDB defaults to the `all-MiniLM-L6-v2` model. On Intel Macs and Apple Silicon Macs, the default embedding function can crash or become extremely slow. Explicitly passing `embedding_function` (even specifying `all-MiniLM-L6-v2` itself) usually sidesteps the issue.
</div>

In [39]:
#!pip install chromadb
#!pip install sentence_transformers

from chromadb.utils import embedding_functions

# Crea la función para calcular embeddings que puede ser pasada como parámetro
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
#emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L12-v2") # Posterior

ChromaDB supports a hosted service, but it also runs locally with a persistent client — avoiding the 8-10+ minute embedding-regeneration cost on every session restart.

The next cell creates the client and tries to load an existing `libros` collection. If the collection doesn't exist yet, it's created fresh and populated with every book's summary.

In [40]:
import chromadb
# Así se crearía un cliente no persistente
#client = chromadb.Client()

# Crea el cliente en la carpeta books_ebd
client = chromadb.PersistentClient(path="books_ebd")

# Intenta acceder a la colección especificando la función que 
# se utiliza para crear los embeddings. Si no puede, la crea.
try:
    #collection = client.get_collection("libros")    
    collection = client.get_collection("libros", embedding_function = emb_fn)
except:
    #collection = client.create_collection("libros")  
    collection = client.create_collection("libros", embedding_function = emb_fn)    
    # añade todos los documentos.
    for identifier, row in df_goodreads.iterrows():
        collection.add(
            documents = row['overview'],         # Textos
            ids = str(identifier),               # Identificador (debe ser un Strng)              
            metadatas={"title": row['title'],    # Campos
                       "author": row['authors']} 
        )   

Query the collection for the 10 books most similar to *Twilight*.

In [41]:
results = collection.query(
    query_texts=df_goodreads.loc[twilight_id]['overview'],
    n_results=10
)

Inspect the `results` structure. Extract and display the title and author for each of the 10 results.

In [46]:
similar_ids = [int(i) for i in results['ids'][0]]

df_goodreads.loc[similar_ids, ['title', 'authors']]



,title,authors
gr_book_id,,
41865,"Twilight (Twilight, #1)",Stephenie Meyer
49041,"New Moon (Twilight, #2)",Stephenie Meyer
6441654,"Twilight and Philosophy: Vampires, Vegetarians...","Rebecca Housel, J. Jeremy Wisnewski, William I..."
11450360,"Twilight: The Graphic Novel, Vol. 2 (Twilight...","Stephenie Meyer, Young Kim"
7619292,"Twilight: The Graphic Novel, Vol. 1 (Twilight:...","Young Kim, Stephenie Meyer"
7140220,Twilight and History,Nancy Reagin
690926,"The Twilight Collection (Twilight, #1-3)",Stephenie Meyer
8726744,The Twilight Saga Complete Collection (Twilig...,Stephenie Meyer
3090465,"The Twilight Saga (Twilight, #1-4)","Stephenie Meyer, Ilyana Kadushin, Matt Walters"


Repeat the query for *Animal Farm* (ID stored in `animalfarm_id`).

In [47]:
results = collection.query(
    query_texts=df_goodreads.loc[animalfarm_id]['overview'],
    n_results=10
)

similar_ids = [int(i) for i in results['ids'][0]]

df_goodreads.loc[similar_ids, ['title', 'authors']]



,title,authors
gr_book_id,,
7613,Animal Farm,George Orwell
5472,Animal Farm / 1984,"George Orwell, Christopher Hitchens"
66933,The Wretched of the Earth,"Frantz Fanon, Jean-Paul Sartre, Richard Philco..."
105703,Pride of Baghdad,"Brian K. Vaughan, Niko Henrichon"
428,Play It as It Lays,"Joan Didion, David Thomson"
11107244,The Better Angels of Our Nature: Why Violence ...,Steven Pinker
93269,Earth Abides,George R. Stewart
5805,V for Vendetta,"Alan Moore, David Lloyd"
29340182,Shrill: Notes from a Loud Woman,Lindy West


The embedding-based matches are thematically closer than TF-IDF — they capture the political-allegory genre that TF-IDF missed.

<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section4"></a>
## 4. Content-Based Recommendation

Content-based recommenders broadly follow three steps:

1. Identify which items the user likes
2. Find items similar to those
3. Rank the candidates

To demonstrate, we pick a random user's activity and build a personalized recommendation from it.

The `./data/ratings.csv` file holds nearly 1 million `(user_id, gr_book_id, rating)` entries from 53,000+ users across 10,000 books. The sheer size makes a traditional user-book matrix expensive to build — we stay in long-form throughout.

In [48]:
df_ratings = pd.read_csv('./data/ratings.csv', sep='\t')
display(df_ratings.head())
df_ratings.shape

,user_id,gr_book_id,rating
0,313,2767052,5
1,438,2767052,3
2,587,2767052,5
3,1168,2767052,4
4,1184,2767052,4


(981756, 3)

<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section41"></a>
### Selecting the user's preferred books

First identify which books the user has positively rated. We pick a random user, filter for their ratings, and keep only those rated 4 or higher.

In [50]:
np.random.seed(0)
test_user = np.random.randint(max(df_ratings['user_id']))

ratings_user = df_ratings[df_ratings['user_id'] == test_user]
ratings_user = ratings_user[ratings_user['rating'] >= 4]
ratings_user

,user_id,gr_book_id,rating
53101,2732,59980,5
209776,2732,19030845,5
214080,2732,16992,5
242344,2732,59952,4
369792,2732,11734,4
401262,2732,16982,4
425738,2732,101911,5
476351,2732,543339,4
524632,2732,97898,5
543609,2732,141372,4


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section42"></a>
### Finding similar books

Save the `gr_book_id` of the user's positively rated books in a Series `books_user`. Preview those books (look up the corresponding rows in `df_goodreads`).

In [51]:
books_user = ratings_user['gr_book_id']
df_goodreads.loc[books_user]

,authors,original_publication_year,title,average_rating,overview,tags
gr_book_id,,,,,,
59980,"Frank Miller, David Mazzucchelli, Richmond Lew...",1987.0,Batman: Year One,4.23,Lieutenant James Gordon takes up a new post in...,cómics graphic-novel batman fiction comics-gra...
19030845,Frank Miller,1986.0,Batman: The Dark Knight Returns #1,4.21,This masterpiece of comics storytelling brings...,comics graphic-novels cómics graphic-novel bat...
16992,"Mark Waid, Alex Ross, Elliot S. Maggin",1996.0,Kingdom Come,4.24,"Writer Mark Waid, coming from his popular work...",comics graphic-novels graphic-novel dc còmics ...
59952,"Frank Miller, Lynn Varley",1998.0,300,3.93,The army of Persia - a force so vast it shakes...,graphic-novels comics cómics graphic-novel his...
11734,Philip Roth,2000.0,"The Human Stain (The American Trilogy, #3)",3.84,"It is 1998, the year in which America is whipp...",fiction novels american contemporary literatur...
16982,"Kurt Busiek, Alex Ross",1993.0,Marvels,4.21,"Welcome to New York. Here, burning figures roa...",comics graphic-novels graphic-novel marvel fic...
101911,A. Manette Ansay,1994.0,Vinegar Hill,3.36,"In a stark, troubling, yet ultimately triumpha...",fiction oprah contemporary-fiction contemporar...
543339,Michael Blake,1988.0,"Dances with Wolves (Dances with Wolves, #1)",4.19,"Ordered to hold an abandoned army post, John D...",historical-fiction fiction western historical ...
97898,Alexander McCall Smith,2003.0,The Full Cupboard of Life (No. 1 Ladies' Detec...,4.02,"Precious Ramotswe, of the No.1 Ladies Detectiv...",mystery fiction africa series mysteries crime ...


Most of these are comics, with at least one horror title mixed in.

For each of the user's liked books, retrieve the 3 most similar books. We can do this in a single ChromaDB query by passing the list of overviews as the `query_texts`. Store the raw result in `results`.

In [52]:
overviews_user = df_goodreads.loc[books_user]['overview'].tolist()

results = collection.query(
    query_texts=overviews_user,
    n_results=3
)

results

{'ids': [['59980', '13223349', '106076'],
  ['19030845', '59960', '107029'],
  ['16992', '616318', '105973'],
  ['59952', '1305', '36'],
  ['11734', '11093329', '61264'],
  ['16982', '953260', '52640'],
  ['101911', '22544024', '534255'],
  ['543339', '11768', '10920'],
  ['97898', '7058', '7036'],
  ['141372', '50', '551536'],
  ['107821', '6303733', '25893709'],
  ['2872', '15705011', '17340050'],
  ['10618', '13516846', '33896'],
  ['28351', '469571', '543873']],
 'embeddings': None,
 'documents': [["Lieutenant James Gordon takes up a new post in the crime-ridden and corrupt city of Gotham, while billionaire Bruce Wayne returns to the scene of his parents' deaths, intent on punishing the criminal element.Collects BATMAN #404-407.",
   "Following his ground-breaking, critically acclaimed run on Detective Comics, writer Scott Snyder (American Vampire) alongside artist Greg Capullo (Spawn)\xa0begins a new era of The Dark Knight as with the relaunch of\xa0Batman, as\xa0a part of DC Comi

### Extracting results

The JSON returned in `results` nests the neighbors per query book. Build four flat lists: `books_ids`, `books_distances`, `book_titles`, and `book_authors`. Cast IDs to integers (they come back as strings).

<div class="alert alert-block alert-warning">
⚠️ `itertools.chain` is a clean way to flatten the per-query result lists.
</div>

In [54]:
import itertools

book_ids = list(map(lambda book_id: int(book_id), itertools.chain(*results['ids'])))

books_distances = list(itertools.chain.from_iterable(results['distances']))

book_titles = list(itertools.chain.from_iterable(
    [ [meta['title'] for meta in group] for group in results['metadatas'] ]
))

book_authors = list(itertools.chain.from_iterable(
    [ [meta['author'] for meta in group] for group in results['metadatas'] ]
))

Build a `results_user` DataFrame from the four lists with columns `book_id`, `authors`, `title`, `distance`. Deduplicate, drop books the user has already read (`book_user`), and sort by ascending distance (closest first).

In [57]:
results_user = pd.DataFrame({
    'book_id': book_ids,
    'authors': book_authors,
    'title': book_titles,
    'distance': books_distances
})

results_user = results_user.drop_duplicates(subset='book_id')

results_user = results_user[~results_user['book_id'].isin(books_user)]

results_user = results_user.sort_values(by='distance')

results_user.head(10)


,book_id,authors,title,distance
25,7058,Alexander McCall Smith,The Good Husband of Zebra Drive (No. 1 Ladies'...,0.417356
26,7036,Alexander McCall Smith,The Kalahari Typing School for Men (No. 1 Ladi...,0.428726
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824
34,15705011,Tracy Chevalier,The Last Runaway,0.771933
5,107029,"Bill Finger, Gardner F. Fox, Bob Kane, Jerry R...","The Batman Chronicles, Vol. 1",0.779259
31,6303733,"Richard Russo, Arthur Morey",That Old Cape Magic,0.786344
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997


<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section43"></a>
### Ranking and final recommendation

We now have the set of candidate books most similar to what the user already liked. To rank, one option is the community `average_rating` — recommend the most universally well-liked candidates.

Take the top 10 candidates by `average_rating` from `results_user`, showing only `title` and `author`. Store the result in a DataFrame named `recomendation`.

In [60]:
results_user['book_id'] = results_user['book_id'].astype(int)
results_user['average_rating'] = results_user['book_id'].map(df_goodreads['average_rating'])
recommendation = results_user.sort_values('average_rating', ascending=False).head(10)[['title', 'authors']]
recommendation


,title,authors
11,The Lord of the Rings: Weapons and Warfare,"Chris Smith, Christopher Lee, Richard Taylor"
10,Gates of Fire: An Epic Novel of the Battle of ...,Steven Pressfield
35,"Losing Hope (Hopeless, #2)",Colleen Hoover
1,"Batman, Volume 1: The Court of Owls","Scott Snyder, Greg Capullo, Jonathan Glapion"
4,Batman: The Dark Knight Returns (The Dark Knig...,"Frank Miller, Klaus Janson, Lynn Varley"
19,Secrets of a Charmed Life,Susan Meissner
7,"Justice, Volume 1","Jim Krueger, Alex Ross, Doug Braithwaite"
17,Weaveworld,Clive Barker
2,Batman: Dark Victory,"Jeph Loeb, Tim Sale"
25,The Good Husband of Zebra Drive (No. 1 Ladies'...,Alexander McCall Smith


The top 10 are thematically close to the user's preferences — almost all comics, matching their history.

<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---

<a id="section5"></a>
## 5. Hybrid Recommender System

Above we ranked candidates by community average rating. An alternative is to rank using collaborative-filtering scores — what the user is *predicted* to rate each candidate. This mixes both worlds: content-based retrieval to narrow the candidate set, then collaborative filtering to personalize the ranking.

Collaborative filtering libraries for Python are surprisingly thin on the ground. The most mature is [surprise](http://surpriselib.com/), though its GitHub activity has slowed recently. We use its SVD implementation for matrix factorization.

<div class="alert alert-block alert-info">
ℹ️ Surprise's author wrote an excellent series of [blog posts](http://nicolas-hug.com/blog/) explaining the basic SVD algorithm — highly recommended reading.
</div>

Load the ratings into a `surprise` dataset structure `ratings_data` and train an SVD matrix factorization model.

In [ ]:
#!pip install surprise


In [61]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection.split import train_test_split

reader = Reader()
ratings_data = Dataset.load_from_df(df_ratings, reader).build_full_trainset()
svd = SVD(n_factors=10)
svd.fit(ratings_data);

Add a `score` column to `results_user` holding the predicted rating for each candidate book for our target user (stored in `test_user`).

In [62]:
results_user['score'] = results_user['book_id'].apply(lambda book_id: svd.predict(test_user, book_id).est)
results_user


,book_id,authors,title,distance,average_rating,score
25,7058,Alexander McCall Smith,The Good Husband of Zebra Drive (No. 1 Ladies'...,0.417356,4.06,3.971921
26,7036,Alexander McCall Smith,The Kalahari Typing School for Men (No. 1 Ladi...,0.428726,3.99,3.974358
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824,4.25,4.339786
34,15705011,Tracy Chevalier,The Last Runaway,0.771933,3.78,3.833591
5,107029,"Bill Finger, Gardner F. Fox, Bob Kane, Jerry R...","The Batman Chronicles, Vol. 1",0.779259,3.99,3.906528
31,6303733,"Richard Russo, Arthur Morey",That Old Cape Magic,0.786344,3.30,3.596056
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744,4.30,4.282216
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644,4.11,4.057657
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884,4.17,4.065316
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997,4.40,4.354093


Take the top 10 books by predicted `score` into a `recommendation` DataFrame and display them.

In [63]:
recommendation = results_user.sort_values('score', ascending=False).head(10)
recommendation


,book_id,authors,title,distance,average_rating,score
11,36,"Chris Smith, Christopher Lee, Richard Taylor",The Lord of the Rings: Weapons and Warfare,1.060990,4.53,4.424137
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997,4.40,4.354093
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824,4.25,4.339786
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744,4.30,4.282216
35,17340050,Colleen Hoover,"Losing Hope (Hopeless, #2)",0.955991,4.39,4.115711
17,52640,Clive Barker,Weaveworld,1.030487,4.13,4.108689
7,616318,"Jim Krueger, Alex Ross, Doug Braithwaite","Justice, Volume 1",0.988655,4.14,4.067031
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884,4.17,4.065316
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644,4.11,4.057657
20,534255,Lucy Grealy,Autobiography of a Face,0.902115,3.96,4.029812


The list is dominated by comics again — consistent with the user's history — but the specific ranking differs slightly from the average-rating approach, reflecting the personalized SVD scores.

<div style="text-align: right">
<a href="#indice">[back to contents]</a>
</div>

---